# Day 21 — Full A/B Test Simulation
## Mini Project: E-commerce Conversion Rate Optimization

> **Business context:** An e-commerce company redesigned its product page (variant B). Does the new design increase the conversion rate and revenue per user?

## Experiment Design

| | Control (A) | Treatment (B) |
|--|--|--|
| Design | Original page | Redesigned page |
| Users | 1,000 | 1,000 |
| Metric 1 | Conversion rate | Conversion rate |
| Metric 2 | Revenue per user | Revenue per user |
| H₀ | No difference | No difference |
| α | 0.05 | 0.05 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
sns.set_theme(style='whitegrid')
np.random.seed(42)

# Load data
df = pd.read_csv('../data/ab_test_data.csv')
control   = df[df['group'] == 'control']
treatment = df[df['group'] == 'treatment']

print("=" * 50)
print("  A/B TEST — EXPERIMENT SUMMARY")
print("=" * 50)
print(f"  Control   users    : {len(control)}")
print(f"  Treatment users    : {len(treatment)}")
print(f"  Control conv. rate : {control['converted'].mean():.3%}")
print(f"  Treatment conv. rate: {treatment['converted'].mean():.3%}")
print(f"  Control avg revenue : ${control['revenue'].mean():.2f}")
print(f"  Treatment avg revenue: ${treatment['revenue'].mean():.2f}")
print("=" * 50)


In [ ]:
# ── Test 1: Conversion Rate (two-proportion z-test) ──────────────────────────
count = [treatment['converted'].sum(), control['converted'].sum()]
nobs  = [len(treatment), len(control)]
z1, p1 = proportions_ztest(count, nobs)
ci_ctrl = proportion_confint(control['converted'].sum(),   len(control),   alpha=0.05)
ci_trt  = proportion_confint(treatment['converted'].sum(), len(treatment), alpha=0.05)

print("TEST 1: Conversion Rate")
print(f"  z = {z1:.4f},  p = {p1:.6f}")
print(f"  Control CI  : ({ci_ctrl[0]:.3%}, {ci_ctrl[1]:.3%})")
print(f"  Treatment CI: ({ci_trt[0]:.3%},  {ci_trt[1]:.3%})")
print(f"  {'✅ Reject H₀ — significant lift in conversions' if p1<0.05 else '❌ No significant difference'}")
print()

# ── Test 2: Revenue per user (t-test) ────────────────────────────────────────
t2, p2 = stats.ttest_ind(treatment['revenue'].values, control['revenue'].values)
n1, n2 = len(treatment), len(control)
v1, v2 = treatment['revenue'].var(ddof=1), control['revenue'].var(ddof=1)
pooled  = np.sqrt(((n1-1)*v1 + (n2-1)*v2) / (n1+n2-2))
d = (treatment['revenue'].mean() - control['revenue'].mean()) / pooled

print("TEST 2: Revenue per User (t-test)")
print(f"  t = {t2:.4f},  p = {p2:.6f}")
print(f"  Cohen's d = {d:.4f}  ({'small' if abs(d)<0.5 else 'medium' if abs(d)<0.8 else 'large'} effect)")
print(f"  {'✅ Reject H₀ — significant revenue difference' if p2<0.05 else '❌ No significant difference'}")


In [ ]:
# ── Full Dashboard ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

CTRL_CLR, TRTM_CLR = '#4C72B0', '#DD8452'
ctrl_r = control['revenue'].values
trtm_r = treatment['revenue'].values

# 1. Conversion rate bar
ax1 = fig.add_subplot(gs[0, 0])
rates  = [control['converted'].mean(), treatment['converted'].mean()]
cis    = [ci_ctrl, ci_trt]
for i, (r, ci, c, lbl) in enumerate(zip(rates, cis, [CTRL_CLR,TRTM_CLR], ['Control','Treatment'])):
    ax1.bar(i, r, color=c, alpha=0.7, width=0.4)
    ax1.errorbar(i, r, yerr=[[r-ci[0]],[ci[1]-r]], fmt='none', color='black', capsize=10, capthick=2)
    ax1.text(i, ci[1]+0.003, f'{r:.2%}', ha='center', fontweight='bold', fontsize=10)
ax1.set_xticks([0,1]); ax1.set_xticklabels(['Control','Treatment'])
ax1.set_title(f'Conversion Rate
p={p1:.4f}', fontweight='bold')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0%}'))

# 2. Revenue distributions
ax2 = fig.add_subplot(gs[0, 1])
nonzero_c = ctrl_r[ctrl_r > 0]
nonzero_t = trtm_r[trtm_r > 0]
for data, lbl, c in zip([nonzero_c, nonzero_t], ['Control','Treatment'], [CTRL_CLR,TRTM_CLR]):
    ax2.hist(data, bins=25, alpha=0.5, color=c, density=True, label=lbl)
    xd = np.linspace(data.min(), data.max(), 200)
    ax2.plot(xd, stats.gaussian_kde(data)(xd), color=c, lw=2)
    ax2.axvline(data.mean(), color=c, linestyle='--', lw=1.5)
ax2.set_title('Revenue Distribution
(Converted users only)')
ax2.legend(); ax2.set_xlabel('Revenue ($)')

# 3. Revenue per user boxplot (all users incl. $0)
ax3 = fig.add_subplot(gs[0, 2])
ax3.boxplot([ctrl_r, trtm_r], labels=['Control','Treatment'],
            patch_artist=True,
            boxprops=dict(facecolor='#AEC6CF', color='navy'),
            medianprops=dict(color='red', lw=2))
ax3.set_title(f'Revenue per User
p={p2:.4f}', fontweight='bold')
ax3.set_ylabel('Revenue ($)')

# 4. p-value visualization for conversion test
ax4 = fig.add_subplot(gs[1, 0])
x = np.linspace(-5, 5, 400)
y = stats.norm.pdf(x)
crit = 1.96
ax4.plot(x, y, 'k-', lw=2)
ax4.fill_between(x, y, where=(x>=crit)|(x<=-crit), alpha=0.3, color='red', label='α=0.05')
ax4.axvline( z1, color='navy', lw=2, linestyle='--', label=f'z={z1:.2f}')
ax4.axvline(-z1, color='navy', lw=2, linestyle='--')
ax4.set_title('Conversion z-test'); ax4.legend()

# 5. Cumulative conversion rate over time
ax5 = fig.add_subplot(gs[1, 1:])
for grp, data, c, lbl in zip([control, treatment], [control, treatment], [CTRL_CLR,TRTM_CLR], ['Control','Treatment']):
    cumulative = data.reset_index(drop=True)['converted'].expanding().mean()
    ax5.plot(cumulative.index, cumulative, color=c, lw=2, alpha=0.8, label=lbl)
ax5.set_title('Cumulative Conversion Rate Over Users')
ax5.set_xlabel('Users processed')
ax5.set_ylabel('Conversion rate')
ax5.legend()
ax5.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0%}'))

fig.suptitle('E-commerce A/B Test — Full Results Dashboard', fontsize=15, fontweight='bold')
plt.savefig('../results/07_ab_test_simulation.png', dpi=150, bbox_inches='tight')
plt.show()


## Business Decision

| Metric | Control | Treatment | Lift | Significant? |
|--------|---------|-----------|------|--------------|
| Conversion Rate | ~10% | ~13% | +3pp | ✅ Yes |
| Revenue/User | ~$5.50 | ~$7.50 | +$2 | ✅ Yes |

> **Recommendation:** Roll out the new product page design to 100% of users.
> The redesign shows a statistically significant improvement in both conversion rate and revenue per user at the 5% significance level.

In [ ]:
# ── Final Summary Report ──────────────────────────────────────────────────────
import sys
sys.path.insert(0, '../src')
from stats_utils import print_test_result

print("\n" + "="*60)
print("  FINAL A/B TEST REPORT")
print("="*60)
print(f"  Experiment      : Product Page Redesign")
print(f"  Sample size     : {len(df)} users ({len(control)} per group)")
print()
print(f"  METRIC 1 — Conversion Rate")
print(f"    Control       : {control['converted'].mean():.2%}")
print(f"    Treatment     : {treatment['converted'].mean():.2%}")
print(f"    Lift          : {treatment['converted'].mean()-control['converted'].mean():+.2%}")
print(f"    p-value       : {p1:.6f}  →  {'SIGNIFICANT' if p1<0.05 else 'NOT SIGNIFICANT'}")
print()
print(f"  METRIC 2 — Revenue per User")
print(f"    Control       : ${control['revenue'].mean():.2f}")
print(f"    Treatment     : ${treatment['revenue'].mean():.2f}")
print(f"    Lift          : ${treatment['revenue'].mean()-control['revenue'].mean():+.2f}")
print(f"    p-value       : {p2:.6f}  →  {'SIGNIFICANT' if p2<0.05 else 'NOT SIGNIFICANT'}")
print()
print("  DECISION: Deploy Treatment (B) to production ✅")
print("="*60)
